In [1]:
import os
import time
import numpy as np
from pathlib import Path
from scipy.io import savemat

from pid.optimizers.tilde_pid import exact_tilde_union_info_minimizer
from pid.optimizers.thin_pid import exact_thin_pid_minimizer
from pid.utils import generate_randomBC

In [2]:
step=100
start=500
max_dX=1000
num_runs=1
ratio=[0.8, 0.1]

In [3]:
# Generate dx, dy, dT sequence
R = np.array([1, ratio[0]])
sequence = np.arange(start, max_dX + 1, step)
XY_dim = np.array([seq * R for seq in sequence]).astype(int)  # TXY_dim = sequence * R = [dT, dx, dy]
print(XY_dim)

[[ 500  400]
 [ 600  480]
 [ 700  560]
 [ 800  640]
 [ 900  720]
 [1000  800]]


In [4]:
dtype = [('dT', int), ('dx', int), ('dy', int), ('obj_tilde', float), ('obj_thin', float),
             ('execution_time_tilde', float), ('execution_time_thin', float), ('itr_tilde', int),
             ('itr_thin', int)]
result_comparison = np.zeros((len(sequence), num_runs), dtype=dtype)  # compare tilde and thin pid

In [5]:
for i, seq in enumerate(sequence):
    dx, dy = XY_dim[i]
    dT = 1
    print(f"\nRunning config [dx, dy, dT]: {dx:3d}-{dy:3d}-{dT:3d}")

    for j in range(num_runs):

        hx, hy = generate_randomBC(dT, dx, dy)
        if dx + dy < 1000:  # only run the original algorithm when the size is small, otherwise it takes too long
            start_time1 = time.time()
            sig_tilde, obj_tilde, itr_tilde = exact_tilde_union_info_minimizer(hx, hy, ret_obj=True, reg=1e-8)
            end_time1 = time.time()
        else:
            sig_tilde, obj_tilde, itr_tilde = 0, 0, 0
            end_time1, start_time1 = 0.0, 0.0
            # print(f"Skipping tilde pid for {dx:3d}-{dy:3d}-{dT:3d}")

        start_time2 = time.time()
        sig_thin, obj_thin, itr_thin = exact_thin_pid_minimizer(hx, hy, ret_obj=True, reg=1e-8)
        end_time2 = time.time()

        execution_time_tilde = end_time1 - start_time1
        execution_time_thin = end_time2 - start_time2
        result_comparison[i, j] = (
            dT, dx, dy, obj_tilde, obj_thin, execution_time_tilde, execution_time_thin, itr_tilde, itr_thin
        )

        # print(f"obj_tilde: {obj_tilde:4.9f}, obj_thin: {obj_thin:4.9f}, time_tilde: {execution_time_tilde:3.4f}, time_thin: {execution_time_thin:3.4f}")

    # print average results
    print(f"Average obj_tilde: {np.mean(result_comparison[i]['obj_tilde']):4.9f}, obj_thin: {np.mean(result_comparison[i]['obj_thin']):4.9f}, time_tilde: {np.mean(result_comparison[i]['execution_time_tilde']):3.4f}, time_thin: {np.mean(result_comparison[i]['execution_time_thin']):3.4f}")


Running config [dx, dy, dT]: 500-400-  1
Average obj_tilde: 4.492788350, obj_thin: 4.492788357, time_tilde: 44.3290, time_thin: 47.5951

Running config [dx, dy, dT]: 600-480-  1
Average obj_tilde: 0.000000000, obj_thin: 4.650844941, time_tilde: 0.0000, time_thin: 89.8640

Running config [dx, dy, dT]: 700-560-  1
Average obj_tilde: 0.000000000, obj_thin: 4.722323744, time_tilde: 0.0000, time_thin: 129.1910

Running config [dx, dy, dT]: 800-640-  1
Average obj_tilde: 0.000000000, obj_thin: 4.883335404, time_tilde: 0.0000, time_thin: 179.0965

Running config [dx, dy, dT]: 900-720-  1
Average obj_tilde: 0.000000000, obj_thin: 4.885685528, time_tilde: 0.0000, time_thin: 304.1465

Running config [dx, dy, dT]: 1000-800-  1
Average obj_tilde: 0.000000000, obj_thin: 4.996573267, time_tilde: 0.0000, time_thin: 339.0747
